# XGBoost Fraud Detection Notebook

In [12]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")



## 1. Configuration

In [13]:
# =========================================================
# 1) Configuration
# =========================================================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "EllipticPlusPlus" / "Transactions Dataset"

TX_FEATURES_PATH = DATA_DIR / "txs_features.csv"
TX_CLASSES_PATH = DATA_DIR / "txs_classes.csv"
TX_EDGES_PATH = DATA_DIR / "txs_edgelist.csv"

ADDR_TX_PATH = DATA_DIR / "AddrTx_edgelist.csv"
TX_ADDR_PATH = DATA_DIR / "TxAddr_edgelist.csv"
WALLET_FEATURES_PATH = DATA_DIR / "wallets_features.csv"

TXID_COL = "txId"
TIME_COL = "Time step"
CLASS_COL = "class"

# Final test window: 40-49
FINAL_TRAIN_END_STEP = 39

# Rolling training window
LOOKBACK_STEPS = 10

# Time-decay weighting
TIME_DECAY = 0.25

# Threshold selection strategy
TARGET_RECALL = 0.95
MIN_VALID_PRECISION = 0.70
THRESHOLD_POLICY = "recent_fold"   # "recent_fold" or "median"
FINAL_THRESHOLD_OVERRIDE = None    # Example: 0.70; use None for automatic selection

# Whether to use transaction graph features
USE_TX_GRAPH_FEATURES = True

# Whether to use wallet-level aggregated features
USE_WALLET_AGG_FEATURES = True

# Whether to use selected lightweight anti-drift features
USE_LIGHT_DRIFT_FEATURES = True

ROLLING_WINDOWS = [
    (24, 29),
    (29, 34),
    (34, 39),
]

THRESHOLD_SCAN_VALUES = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

RANDOM_STATE = 42
MODEL_OUT = "ellipticpp_xgb_with_wallet_features.joblib"

BASE_XGB_PARAMS = {
    "n_estimators": 4000,
    "max_depth": 4,
    "learning_rate": 0.03,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "min_child_weight": 10,
    "gamma": 0.0,
    "reg_lambda": 2.0,
    "reg_alpha": 0.5,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
    "early_stopping_rounds": 200,
}



## 2. Utility Functions

In [14]:
# =========================================================
# 2) Utility functions
# =========================================================
def safe_roc_auc(y_true, y_prob):
    try:
        return roc_auc_score(y_true, y_prob)
    except Exception:
        return np.nan


def safe_pr_auc(y_true, y_prob):
    try:
        return average_precision_score(y_true, y_prob)
    except Exception:
        return np.nan


def calc_scale_pos_weight(y):
    pos = int((y == 1).sum())
    neg = int((y == 0).sum())
    return neg / max(pos, 1)


def make_time_decay_weights(time_steps, max_step, decay=0.25):
    return np.exp(decay * (time_steps - max_step))


def print_basic_stats(df, name):
    pos = int((df["label"] == 1).sum())
    neg = int((df["label"] == 0).sum())
    rate = pos / max(len(df), 1)
    print(f"{name}: n={len(df)}, pos={pos}, neg={neg}, pos_rate={rate:.4f}")


def build_threshold_table(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.01, 0.96, 0.01)

    rows = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        rows.append([
            float(t),
            precision_score(y_true, y_pred, zero_division=0),
            recall_score(y_true, y_pred, zero_division=0),
            f1_score(y_true, y_pred, zero_division=0),
            int(y_pred.sum()),
        ])

    return pd.DataFrame(
        rows,
        columns=["threshold", "precision", "recall", "f1", "alert_count"],
    )


def choose_threshold_from_valid(
    threshold_table,
    target_recall=0.95,
    min_precision=0.70,
):
    # 1) Prefer thresholds that satisfy both the recall target and the precision floor
    pool1 = threshold_table[
        (threshold_table["recall"] >= target_recall) &
        (threshold_table["precision"] >= min_precision)
    ].copy()

    if len(pool1) > 0:
        pool1 = pool1.sort_values(
            ["precision", "f1", "threshold"],
            ascending=[False, False, False]
        )
        row = pool1.iloc[0]
        return float(row["threshold"]), f"recall>={target_recall:.2f}&precision>={min_precision:.2f}"

    # 2) Otherwise, among thresholds that satisfy the precision floor, choose the best F1
    pool2 = threshold_table[threshold_table["precision"] >= min_precision].copy()
    if len(pool2) > 0:
        pool2 = pool2.sort_values(
            ["f1", "recall", "threshold"],
            ascending=[False, False, False]
        )
        row = pool2.iloc[0]
        return float(row["threshold"]), f"fallback_precision>={min_precision:.2f}_best_f1"

    # 3) As a final fallback, choose the global best F1
    pool3 = threshold_table.sort_values(["f1", "recall"], ascending=[False, False])
    row = pool3.iloc[0]
    return float(row["threshold"]), "fallback_best_f1"


def evaluate_split(name, y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n========== {name} ==========")
    print(f"threshold = {threshold:.4f}")
    print(f"Precision = {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall    = {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1        = {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"ROC-AUC   = {safe_roc_auc(y_true, y_prob):.4f}")
    print(f"PR-AUC    = {safe_pr_auc(y_true, y_prob):.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))


def evaluate_by_time_step(df_eval, threshold):
    rows = []
    for step, g in df_eval.groupby(TIME_COL):
        y_true = g["label"].values
        y_prob = g["prob"].values
        y_pred = (y_prob >= threshold).astype(int)

        rows.append([
            int(step),
            len(g),
            int((g["label"] == 1).sum()),
            precision_score(y_true, y_pred, zero_division=0),
            recall_score(y_true, y_pred, zero_division=0),
            f1_score(y_true, y_pred, zero_division=0),
            safe_roc_auc(y_true, y_prob),
            safe_pr_auc(y_true, y_prob),
        ])

    return pd.DataFrame(
        rows,
        columns=[
            "time_step", "n", "pos", "precision", "recall",
            "f1", "roc_auc", "pr_auc"
        ],
    ).sort_values("time_step")



## 3. Load Data

In [15]:
# =========================================================
# 3) Load data
# =========================================================
def load_data():
    tx_features_all = pd.read_csv(TX_FEATURES_PATH)
    tx_classes = pd.read_csv(TX_CLASSES_PATH)
    tx_edges = pd.read_csv(TX_EDGES_PATH)

    addr_tx = pd.read_csv(ADDR_TX_PATH)      # input_address -> txId
    tx_addr = pd.read_csv(TX_ADDR_PATH)      # txId -> output_address
    wallet_features = pd.read_csv(WALLET_FEATURES_PATH)

    df = tx_features_all.merge(tx_classes, on=TXID_COL, how="inner")
    df = df[df[CLASS_COL].isin([1, 2])].copy()
    df["label"] = (df[CLASS_COL] == 1).astype(int)

    return df, tx_features_all, tx_edges, addr_tx, tx_addr, wallet_features



## 4. Optional Transaction Graph Features

In [16]:
# =========================================================
# 4) Optional transaction graph features
# =========================================================
def add_safe_tx_graph_features(df_features_all, df_labeled, df_edges):
    step_map = df_features_all[[TXID_COL, TIME_COL]].drop_duplicates()

    e = df_edges.merge(
        step_map.rename(columns={TXID_COL: "txId1", TIME_COL: "src_step"}),
        on="txId1",
        how="left"
    )
    e = e.merge(
        step_map.rename(columns={TXID_COL: "txId2", TIME_COL: "dst_step"}),
        on="txId2",
        how="left"
    )

    e = e.dropna(subset=["src_step", "dst_step"]).copy()
    e = e[e["src_step"] < e["dst_step"]].copy()

    past_in_degree = e.groupby("txId2").size().rename("past_in_degree")
    past_unique_predecessors = (
        e.groupby("txId2")["txId1"].nunique().rename("past_unique_predecessors")
    )

    e["edge_age_steps"] = e["dst_step"] - e["src_step"]
    mean_input_age_steps = (
        e.groupby("txId2")["edge_age_steps"].mean().rename("mean_input_age_steps")
    )

    df = df_labeled.merge(
        past_in_degree, left_on=TXID_COL, right_index=True, how="left"
    )
    df = df.merge(
        past_unique_predecessors, left_on=TXID_COL, right_index=True, how="left"
    )
    df = df.merge(
        mean_input_age_steps, left_on=TXID_COL, right_index=True, how="left"
    )

    for c in ["past_in_degree", "past_unique_predecessors", "mean_input_age_steps"]:
        df[c] = df[c].fillna(0)

    df["log_past_in_degree"] = np.log1p(df["past_in_degree"])
    df["log_past_unique_predecessors"] = np.log1p(df["past_unique_predecessors"])

    return df



## 5. Wallet Feature Aggregation

In [17]:
# =========================================================
# 5) Main addition: aggregate wallet features back to each transaction
#    Time-safe design: join exactly on (address, Time step)
# =========================================================
def aggregate_side_features(joined_df, txid_col, addr_col, prefix, feature_cols):
    result = joined_df.groupby(txid_col).agg(
        **{
            f"{prefix}_address_count": (addr_col, "nunique"),
            f"{prefix}_row_count": (addr_col, "size"),
        }
    ).reset_index()

    # Match rate: count a row as matched if at least one wallet feature is present
    matched_flag = joined_df[feature_cols].notna().any(axis=1).astype(int)
    tmp = joined_df[[txid_col]].copy()
    tmp[f"{prefix}_matched_flag"] = matched_flag.values

    match_stats = tmp.groupby(txid_col).agg(
        **{
            f"{prefix}_matched_rows": (f"{prefix}_matched_flag", "sum"),
        }
    ).reset_index()

    result = result.merge(match_stats, on=txid_col, how="left")
    result[f"{prefix}_match_rate"] = (
        result[f"{prefix}_matched_rows"] / result[f"{prefix}_row_count"].replace(0, 1)
    )

    # Numeric aggregations
    agg_dict = {}
    for c in feature_cols:
        agg_dict[f"{prefix}_{c}_mean"] = (c, "mean")
        agg_dict[f"{prefix}_{c}_max"] = (c, "max")
        agg_dict[f"{prefix}_{c}_std"] = (c, "std")

    numeric_agg = joined_df.groupby(txid_col).agg(**agg_dict).reset_index()
    result = result.merge(numeric_agg, on=txid_col, how="left")

    return result


def add_wallet_aggregate_features(df, addr_tx, tx_addr, wallet_features):
    # Transaction time steps
    tx_time = df[[TXID_COL, TIME_COL]].drop_duplicates()

    # Select only a compact set of wallet columns to avoid feature explosion
    preferred_wallet_cols = [
        "num_txs_as_sender",
        "num_txs_as_receiver",
        "total_txs",
        "lifetime_in_blocks",
        "first_sent_block",
        "first_received_block",
        "blocks_btwn_output_txs_min",
        "blocks_btwn_output_txs_max",
    ]
    wallet_feature_cols = [c for c in preferred_wallet_cols if c in wallet_features.columns]

    if len(wallet_feature_cols) == 0:
        print("\nWarning: expected numeric wallet features were not found in wallets_features.csv; skipping wallet aggregation.")
        return df, []

    # Drop duplicates to avoid join expansion when the same address/time-step pair appears multiple times
    wallet_features = wallet_features.drop_duplicates(subset=["address", TIME_COL]).copy()

    # -------- Input-address side --------
    in_map = addr_tx.merge(tx_time, on=TXID_COL, how="inner")
    in_join = in_map.merge(
        wallet_features[["address", TIME_COL] + wallet_feature_cols],
        left_on=["input_address", TIME_COL],
        right_on=["address", TIME_COL],
        how="left",
    )

    in_agg = aggregate_side_features(
        joined_df=in_join,
        txid_col=TXID_COL,
        addr_col="input_address",
        prefix="in_wallet",
        feature_cols=wallet_feature_cols,
    )

    # -------- Output-address side --------
    out_map = tx_addr.merge(tx_time, on=TXID_COL, how="inner")
    out_join = out_map.merge(
        wallet_features[["address", TIME_COL] + wallet_feature_cols],
        left_on=["output_address", TIME_COL],
        right_on=["address", TIME_COL],
        how="left",
    )

    out_agg = aggregate_side_features(
        joined_df=out_join,
        txid_col=TXID_COL,
        addr_col="output_address",
        prefix="out_wallet",
        feature_cols=wallet_feature_cols,
    )

    # Merge aggregated features back to the transaction table
    df = df.merge(in_agg, on=TXID_COL, how="left")
    df = df.merge(out_agg, on=TXID_COL, how="left")

    added_cols = [c for c in df.columns if c.startswith("in_wallet_") or c.startswith("out_wallet_")]

    # Add a few input/output difference and ratio features
    cross_added = []
    for c in wallet_feature_cols:
        in_mean = f"in_wallet_{c}_mean"
        out_mean = f"out_wallet_{c}_mean"
        if in_mean in df.columns and out_mean in df.columns:
            diff_col = f"inout_{c}_mean_diff"
            ratio_col = f"inout_{c}_mean_ratio"
            df[diff_col] = df[in_mean] - df[out_mean]
            df[ratio_col] = df[in_mean] / (df[out_mean] + 1e-6)
            cross_added.extend([diff_col, ratio_col])

    if "in_wallet_address_count" in df.columns and "out_wallet_address_count" in df.columns:
        df["inout_address_count_diff"] = (
            df["in_wallet_address_count"] - df["out_wallet_address_count"]
        )
        df["inout_address_count_ratio"] = (
            df["in_wallet_address_count"] / (df["out_wallet_address_count"] + 1)
        )
        cross_added.extend(["inout_address_count_diff", "inout_address_count_ratio"])

    for c in added_cols + cross_added:
        df[c] = df[c].replace([np.inf, -np.inf], np.nan)

    return df, added_cols + cross_added



## 6. Lightweight Anti-Drift Features

In [18]:
# =========================================================
# 6) Lightweight anti-drift features: normalize only a small set of strong features within each step
# =========================================================
def add_light_drift_features(df):
    added = []

    candidate_cols = [
        "size",
        "num_output_addresses",
        "fees",
        "past_in_degree",
        "past_unique_predecessors",
        "in_wallet_total_txs_mean",
        "out_wallet_total_txs_mean",
        "in_wallet_lifetime_in_blocks_mean",
        "out_wallet_lifetime_in_blocks_mean",
        "in_wallet_num_txs_as_sender_mean",
        "out_wallet_num_txs_as_receiver_mean",
    ]
    candidate_cols = [c for c in candidate_cols if c in df.columns]

    for c in candidate_cols:
        step_mean = df.groupby(TIME_COL)[c].transform("mean")
        step_std = df.groupby(TIME_COL)[c].transform("std").replace(0, np.nan).fillna(1.0)

        z_col = f"{c}_step_z"
        rank_col = f"{c}_step_rank_pct"

        df[z_col] = ((df[c] - step_mean) / step_std).replace([np.inf, -np.inf], 0).fillna(0)
        df[rank_col] = df.groupby(TIME_COL)[c].rank(pct=True).replace([np.inf, -np.inf], 0).fillna(0)

        added.extend([z_col, rank_col])

    # A small set of interaction features
    if "size" in df.columns and "num_output_addresses" in df.columns:
        df["size_per_output"] = df["size"] / (df["num_output_addresses"] + 1)
        df["size_x_num_outputs"] = df["size"] * df["num_output_addresses"]
        added.extend(["size_per_output", "size_x_num_outputs"])

    if "fees" in df.columns and "size" in df.columns:
        df["fees_per_size"] = df["fees"] / (df["size"] + 1e-6)
        added.append("fees_per_size")

    for c in added:
        df[c] = df[c].replace([np.inf, -np.inf], 0).fillna(0)

    return df, added


# =========================================================

## 7. Feature Selection Helper

In [19]:
# 7) Feature columns
# =========================================================
def get_feature_columns(df):
    ignore_cols = {TXID_COL, CLASS_COL, "label", TIME_COL}
    return [c for c in df.columns if c not in ignore_cols]



## 8. Rolling Validation

In [20]:
# =========================================================
# 8) Rolling validation
# =========================================================
def run_one_fold(
    df,
    feature_cols,
    train_end_step,
    valid_end_step,
    lookback_steps,
    target_recall,
    min_valid_precision,
    time_decay,
):
    train_df = df[
        (df[TIME_COL] > train_end_step - lookback_steps) &
        (df[TIME_COL] <= train_end_step)
    ].copy()

    valid_df = df[
        (df[TIME_COL] > train_end_step) &
        (df[TIME_COL] <= valid_end_step)
    ].copy()

    X_train = train_df[feature_cols].copy()
    y_train = train_df["label"].copy()

    X_valid = valid_df[feature_cols].copy()
    y_valid = valid_df["label"].copy()

    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train)
    X_valid_imp = imputer.transform(X_valid)

    scale_pos_weight = calc_scale_pos_weight(y_train)
    sample_weight = make_time_decay_weights(
        train_df[TIME_COL].values,
        max_step=train_df[TIME_COL].max(),
        decay=time_decay,
    )

    model = XGBClassifier(
        **BASE_XGB_PARAMS,
        scale_pos_weight=scale_pos_weight,
    )

    model.fit(
        X_train_imp,
        y_train,
        sample_weight=sample_weight,
        eval_set=[(X_valid_imp, y_valid)],
        verbose=False,
    )

    valid_prob = model.predict_proba(X_valid_imp)[:, 1]
    threshold_table = build_threshold_table(y_valid, valid_prob)
    selected_threshold, threshold_mode = choose_threshold_from_valid(
        threshold_table,
        target_recall=target_recall,
        min_precision=min_valid_precision,
    )

    y_valid_pred = (valid_prob >= selected_threshold).astype(int)
    best_iter = getattr(model, "best_iteration", None)
    if best_iter is None:
        best_iter = model.n_estimators - 1

    return {
        "train_end_step": train_end_step,
        "valid_end_step": valid_end_step,
        "best_n_estimators": int(best_iter + 1),
        "selected_threshold": float(selected_threshold),
        "valid_precision": float(precision_score(y_valid, y_valid_pred, zero_division=0)),
        "valid_recall": float(recall_score(y_valid, y_valid_pred, zero_division=0)),
        "valid_f1": float(f1_score(y_valid, y_valid_pred, zero_division=0)),
        "valid_pr_auc": float(safe_pr_auc(y_valid, valid_prob)),
        "threshold_mode": threshold_mode,
    }


def rolling_time_validation(
    df,
    feature_cols,
    rolling_windows,
    lookback_steps,
    target_recall,
    min_valid_precision,
    time_decay,
):
    results = []

    print("\n================ ROLLING TIME VALIDATION ================")
    for i, (train_end_step, valid_end_step) in enumerate(rolling_windows, start=1):
        res = run_one_fold(
            df=df,
            feature_cols=feature_cols,
            train_end_step=train_end_step,
            valid_end_step=valid_end_step,
            lookback_steps=lookback_steps,
            target_recall=target_recall,
            min_valid_precision=min_valid_precision,
            time_decay=time_decay,
        )
        results.append(res)

        print(f"\n[Fold {i}] train<= {train_end_step}, valid=({train_end_step}, {valid_end_step}]")
        print(
            f"best_n_estimators={res['best_n_estimators']}, "
            f"threshold={res['selected_threshold']:.2f}, "
            f"mode={res['threshold_mode']}, "
            f"valid_PR-AUC={res['valid_pr_auc']:.4f}, "
            f"valid_F1={res['valid_f1']:.4f}, "
            f"valid_Recall={res['valid_recall']:.4f}, "
            f"valid_Precision={res['valid_precision']:.4f}"
        )

    summary = pd.DataFrame(results)

    median_threshold = float(summary["selected_threshold"].median())
    recent_threshold = float(summary.iloc[-1]["selected_threshold"])
    robust_n_estimators = int(round(summary["best_n_estimators"].median()))

    print("\n================ ROLLING SUMMARY ================")
    print(summary.to_string(index=False))
    print(f"\nmedian threshold = {median_threshold:.4f}")
    print(f"recent-fold threshold = {recent_threshold:.4f}")
    print(f"robust_n_estimators = {robust_n_estimators}")

    return summary, median_threshold, recent_threshold, robust_n_estimators



## 9. Final Training

In [21]:
# =========================================================
# 9) Final training
# =========================================================
def train_final_model(
    df,
    feature_cols,
    final_train_end_step,
    lookback_steps,
    robust_n_estimators,
    time_decay,
):
    train_df = df[
        (df[TIME_COL] > final_train_end_step - lookback_steps) &
        (df[TIME_COL] <= final_train_end_step)
    ].copy()
    test_df = df[df[TIME_COL] > final_train_end_step].copy()

    X_train = train_df[feature_cols].copy()
    y_train = train_df["label"].copy()
    X_test = test_df[feature_cols].copy()
    y_test = test_df["label"].copy()

    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)

    scale_pos_weight = calc_scale_pos_weight(y_train)
    sample_weight = make_time_decay_weights(
        train_df[TIME_COL].values,
        max_step=train_df[TIME_COL].max(),
        decay=time_decay,
    )

    final_model = XGBClassifier(
        n_estimators=robust_n_estimators,
        max_depth=BASE_XGB_PARAMS["max_depth"],
        learning_rate=BASE_XGB_PARAMS["learning_rate"],
        subsample=BASE_XGB_PARAMS["subsample"],
        colsample_bytree=BASE_XGB_PARAMS["colsample_bytree"],
        min_child_weight=BASE_XGB_PARAMS["min_child_weight"],
        gamma=BASE_XGB_PARAMS["gamma"],
        reg_lambda=BASE_XGB_PARAMS["reg_lambda"],
        reg_alpha=BASE_XGB_PARAMS["reg_alpha"],
        objective=BASE_XGB_PARAMS["objective"],
        eval_metric=BASE_XGB_PARAMS["eval_metric"],
        tree_method=BASE_XGB_PARAMS["tree_method"],
        n_jobs=BASE_XGB_PARAMS["n_jobs"],
        random_state=BASE_XGB_PARAMS["random_state"],
        scale_pos_weight=scale_pos_weight,
    )

    final_model.fit(
        X_train_imp,
        y_train,
        sample_weight=sample_weight,
        verbose=False,
    )

    test_prob = final_model.predict_proba(X_test_imp)[:, 1]

    return {
        "model": final_model,
        "imputer": imputer,
        "train_df": train_df,
        "test_df": test_df,
        "y_test": y_test,
        "test_prob": test_prob,
    }



## 10. Main Workflow and Execution

In [ ]:
# =========================================================
# 10) Main workflow
# =========================================================
def main():
    df, tx_features_all, tx_edges, addr_tx, tx_addr, wallet_features = load_data()

    print("Number of labeled samples (binary classification):", len(df))
    print("Label distribution:")
    print(df["label"].value_counts(dropna=False).sort_index())

    added_cols = []

    if USE_TX_GRAPH_FEATURES:
        df = add_safe_tx_graph_features(tx_features_all, df, tx_edges)
        print("\nAdded transaction graph features.")

    if USE_WALLET_AGG_FEATURES:
        df, wallet_added = add_wallet_aggregate_features(df, addr_tx, tx_addr, wallet_features)
        added_cols.extend(wallet_added)
        print(f"\nAdded wallet aggregation features: {len(wallet_added)} columns.")

    if USE_LIGHT_DRIFT_FEATURES:
        df, drift_added = add_light_drift_features(df)
        added_cols.extend(drift_added)
        print(f"\nAdded lightweight anti-drift features: {len(drift_added)} columns.")

    feature_cols = get_feature_columns(df)
    print(f"\nNumber of features = {len(feature_cols)}")

    pretest_df = df[df[TIME_COL] <= FINAL_TRAIN_END_STEP].copy()
    test_df = df[df[TIME_COL] > FINAL_TRAIN_END_STEP].copy()

    print("\nData overview:")
    print_basic_stats(pretest_df, "pre-test")
    print_basic_stats(test_df, "test")

    rolling_summary, median_threshold, recent_threshold, robust_n_estimators = rolling_time_validation(
        df=pretest_df,
        feature_cols=feature_cols,
        rolling_windows=ROLLING_WINDOWS,
        lookback_steps=LOOKBACK_STEPS,
        target_recall=TARGET_RECALL,
        min_valid_precision=MIN_VALID_PRECISION,
        time_decay=TIME_DECAY,
    )

    if FINAL_THRESHOLD_OVERRIDE is not None:
        final_threshold = float(FINAL_THRESHOLD_OVERRIDE)
        threshold_source = "manual_override"
    else:
        if THRESHOLD_POLICY == "median":
            final_threshold = median_threshold
            threshold_source = "median_threshold"
        else:
            final_threshold = recent_threshold
            threshold_source = "recent_fold_threshold"

    print(f"\nFinal threshold source: {threshold_source}")
    print(f"Final threshold = {final_threshold:.4f}")

    final_pack = train_final_model(
        df=df,
        feature_cols=feature_cols,
        final_train_end_step=FINAL_TRAIN_END_STEP,
        lookback_steps=LOOKBACK_STEPS,
        robust_n_estimators=robust_n_estimators,
        time_decay=TIME_DECAY,
    )

    final_model = final_pack["model"]
    y_test = final_pack["y_test"]
    test_df = final_pack["test_df"].copy()
    test_prob = final_pack["test_prob"]

    evaluate_split("TEST", y_test, test_prob, final_threshold)

    test_eval = test_df[[TXID_COL, TIME_COL, "label"]].copy()
    test_eval["prob"] = test_prob

    step_metrics = evaluate_by_time_step(test_eval, threshold=final_threshold)
    print("\n========== TEST METRICS BY TIME STEP ==========")
    print(step_metrics.to_string(index=False))

    test_threshold_table = build_threshold_table(
        y_test,
        test_prob,
        thresholds=THRESHOLD_SCAN_VALUES,
    )
    print("\n========== TEST THRESHOLD SCAN (ANALYSIS ONLY) ==========")
    print(test_threshold_table.to_string(index=False))

    importances = pd.Series(final_model.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)

    print("\nTop 25 Feature Importances:")
    print(importances.head(25).to_string())

    package = {
        "model": final_model,
        "feature_cols": feature_cols,
        "final_threshold": final_threshold,
        "threshold_source": threshold_source,
        "rolling_summary": rolling_summary,
        "test_step_metrics": step_metrics,
        "test_threshold_table": test_threshold_table,
        "added_feature_cols": added_cols,
    }
    joblib.dump(package, MODEL_OUT)
    print(f"\nModel saved to: {MODEL_OUT}")


if __name__ == "__main__":
    main()

Number of labeled samples (binary classification): 46564
Label distribution:
label
0    42019
1     4545
Name: count, dtype: int64

Added transaction graph features.
